In [49]:
import torch
import torchvision
from torchvision import transforms

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.14.0+cpu
Torchvision version: 0.29.0+cpu
CUDA available: False


In [50]:
from pathlib import Path

DATASET_DIR = Path("../dataset")

TRAIN_DIR = DATASET_DIR / "seg_train"
TEST_DIR = DATASET_DIR / "seg_test"

print("Train directory:", TRAIN_DIR.resolve())
print("Test directory :", TEST_DIR.resolve())

print("\nClasses:")

classes = sorted([
    folder.name
    for folder in TRAIN_DIR.iterdir()
    if folder.is_dir()
])

print(classes)
print("Number of classes:", len(classes))

Train directory: C:\intel-image-classification\dataset\seg_train
Test directory : C:\intel-image-classification\dataset\seg_test

Classes:
['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Number of classes: 6


In [51]:
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split

# First load the training dataset
full_dataset = ImageFolder(TRAIN_DIR)

print("Total training images:", len(full_dataset))
print("Classes:", full_dataset.classes)

Total training images: 14034
Classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [52]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("✅ Transformations created")

✅ Transformations created


In [53]:
from torch.utils.data import DataLoader

# Recreate dataset with training transformations
train_dataset_full = ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

# Calculate split sizes
train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

# Reproducible split
generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    train_dataset_full,
    [train_size, val_size],
    generator=generator
)

# Test dataset
test_dataset = ImageFolder(
    TEST_DIR,
    transform=test_transform
)

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training images  :", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images      :", len(test_dataset))
print("Batch size       :", BATCH_SIZE)

Training images  : 11227
Validation images: 2807
Test images      : 3000
Batch size       : 32


In [54]:
images, labels = next(iter(train_loader))

print("Image batch shape :", images.shape)
print("Label batch shape :", labels.shape)
print("Labels            :", labels[:10])

Image batch shape : torch.Size([32, 3, 224, 224])
Label batch shape : torch.Size([32])
Labels            : tensor([4, 1, 0, 5, 2, 5, 0, 2, 4, 4])


In [55]:
import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import EfficientNet_B0_Weights

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.14.0+cpu
CUDA available: False


In [56]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


Using device: cpu


In [57]:
print("Classes:")

for index, class_name in enumerate(classes):
    print(f"{index} -> {class_name}")

NUM_CLASSES = len(classes)

print("\nNumber of classes:", NUM_CLASSES)

Classes:
0 -> buildings
1 -> forest
2 -> glacier
3 -> mountain
4 -> sea
5 -> street

Number of classes: 6


In [58]:
weights = EfficientNet_B0_Weights.DEFAULT

model = models.efficientnet_b0(
    weights=weights
)

print("✅ EfficientNet-B0 loaded")

✅ EfficientNet-B0 loaded


In [59]:
print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [60]:
in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    in_features,
    NUM_CLASSES
)

print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=6, bias=True)
)


In [61]:
for parameter in model.features.parameters():
    parameter.requires_grad = False

print("✅ EfficientNet feature extractor frozen")
print("✅ Only the classifier will be trained")


✅ EfficientNet feature extractor frozen
✅ Only the classifier will be trained


In [62]:
model = model.to(device)

print("Model device:", device)

Model device: cpu


In [63]:
criterion = nn.CrossEntropyLoss()

print("Loss function:", criterion)

Loss function: CrossEntropyLoss()


In [64]:
optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=0.001
)

print("Optimizer:", optimizer)

Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [65]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters     :", total_params)
print("Trainable parameters :", trainable_params)
print("Frozen parameters    :", total_params - trainable_params)

Total parameters     : 4015234
Trainable parameters : 7686
Frozen parameters    : 4007548


In [66]:
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

outputs = model(images)

print("Input shape :", images.shape)
print("Output shape:", outputs.shape)

Input shape : torch.Size([32, 3, 224, 224])
Output shape: torch.Size([32, 6])


In [67]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [68]:
def validate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [69]:
EPOCHS = 5

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_accuracy = 0.0

print("Epochs:", EPOCHS)
print("Best validation accuracy:", best_val_accuracy)

Epochs: 5
Best validation accuracy: 0.0


In [70]:
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Model directory:", MODEL_DIR.resolve())
print("✅ Models folder ready")

Model directory: C:\intel-image-classification\models
✅ Models folder ready


In [77]:
EPOCHS = 5

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_accuracy = 0.0

for epoch in range(EPOCHS):

    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_accuracy = validate(
        model,
        val_loader,
        criterion,
        device
    )

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_accuracy:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_accuracy:.4f}"
    )

    # Save best model
    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "class_names": classes,
                "image_size": IMAGE_SIZE,
                "model_name": "EfficientNet-B0",
                "num_classes": NUM_CLASSES,
                "val_accuracy": val_accuracy
            },
            MODEL_DIR / "intel_efficientnet_b0_v1_best.pth"
        ) 

        print("✅ Best model saved!")

KeyboardInterrupt: 

In [ ]:
model_path = MODEL_DIR / "intel_efficientnet_b0_v1_best.pth"

print("Path:", model_path.resolve())
print("Exists:", model_path.exists())

if model_path.exists():
    print("Size:", round(model_path.stat().st_size / (1024 * 1024), 2), "MB")

Path: C:\intel-image-classification\models\intel_efficientnet_b0_v1_best.pth
Exists: True
Size: 15.6 MB


In [73]:
model_path = MODEL_DIR / "intel_efficientnet_b0_v1_best.pth"

checkpoint = torch.load(
    model_path,
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("✅ Best model loaded")
print("Validation Accuracy:", checkpoint["val_accuracy"])
print("Classes:", checkpoint["class_names"])


✅ Best model loaded
Validation Accuracy: 0.21588884930530816
Classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [75]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

all_labels = []
all_predictions = []

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.numpy())
        all_predictions.extend(predictions.cpu().numpy())

print("✅ Test prediction completed")
print("Total test images:", len(all_labels))

ModuleNotFoundError: No module named 'sklearn'